# Hand Gesture Recognition — Exploratory Data Analysis (EDA)
Analisis data landmark tangan 21 titik MediaPipe (63 fitur koordinat) untuk klasifikasi angka 0 sampai 5.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from src.preprocess import load_dataset, prepare_features_and_labels, LANDMARK_FEATURE_NAMES

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

## 1. Memuat Dataset dan Memeriksa Distribusi Kelas

In [ ]:
df = load_dataset("data/processed/synthetic_landmarks_demo.csv")
print(f"Total baris: {len(df)}, Total kolom: {df.shape[1]}")
print("\nDistribusi Sampel per Kelas:")
print(df["label"].value_counts().sort_index())

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="label", palette="Blues_d")
plt.title("Distribusi Sampel per Kelas (0–5)", fontweight="bold")
plt.xlabel("Kelas Gestur")
plt.ylabel("Jumlah Sampel")
plt.show()

## 2. Normalisasi Fitur (Wrist-Centering & Scale Invariance)

In [ ]:
X_norm, y = prepare_features_and_labels(df, normalize=True)
print("Bentuk X_norm:", X_norm.shape)
print("Koordinat pergelangan tangan (wrist 0) setelah normalisasi:", X_norm[0, :3])
print("Jarak maksimum dari wrist:", np.max(np.linalg.norm(X_norm.reshape(-1, 21, 3), axis=2)))

## 3. Reduksi Dimensi Menggunakan PCA (2D Projection)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_norm)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Label": [f"Angka {lbl}" for lbl in y]
})

plt.figure(figsize=(9, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Label", palette="tab10", alpha=0.8)
plt.title("Proyeksi PCA 2D Landmark Gestur Tangan", fontweight="bold")
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% varians)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% varians)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()